# LBPH 人脸识别训练、测试与评估工作流

这个 Notebook 是 `LBPH/src` 模块的可视化编排层。核心训练、推理、评估、后端 artifact 转换和 benchmark 打包逻辑仍保留在模块中，避免 Notebook 与评分系统入口产生两套实现。

## 1. 环境初始化

从仓库根目录或 `LBPH` 目录启动 Jupyter 都可以。下面的代码会自动定位 `LBPH` 工作区，并将其加入 `sys.path`。

In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / "src").exists():
    candidate = WORKSPACE / "LBPH"
    if (candidate / "src").exists():
        WORKSPACE = candidate.resolve()

if not (WORKSPACE / "src").exists():
    raise RuntimeError("请从仓库根目录或 LBPH 目录启动 Notebook")

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.backend_adapter import convert_algorithm_to_backend_artifacts
from src.benchmark_pack import create_benchmark_package
from src.dataset import read_manifest, scan_faces_raw, stratified_split
from src.evaluate import evaluate_directory
from src.predict import LBPHPredictor
from src.preprocess import BACKEND_COMPAT_PREPROCESS_CONFIG, preprocess_image
from src.staged_train import load_training_state, run_training_stage
from src.train import train_lbph
from src.training_tools import grid_search_lbph, run_threshold_sweep

display(Markdown(f"**LBPH workspace:** `{WORKSPACE}`"))

## 2. 全局配置

默认不自动执行会写入模型或报告的步骤。确认数据已放入 `TestData/Faces_raw/<identity>/*.jpg` 后，可以把对应开关改为 `True`。

In [ ]:
RAW_DIR = "TestData/Faces_raw"
TRAIN_DIR = "TestData/Faces_train"
VAL_DIR = "TestData/Faces_val"
TEST_DIR = "TestData/Faces_test"
MANIFEST_PATH = "metadata/manifest.csv"
ALGORITHM_DIR = "Algorithm"
REPORTS_DIR = "reports"
BACKEND_ARTIFACT_DIR = "backend_artifacts/lbph-opencv-confidence-v1"
BENCHMARK_ZIP = "benchmark_packages/lbph_benchmark_package.zip"

RUN_SCAN_RAW = False
RUN_SPLIT = False
RUN_TRAIN = False
RUN_STAGED_TRAINING = False
RUN_EVALUATE = False
RUN_BACKEND_EXPORT = False
RUN_BENCHMARK_PACKAGE = False
RUN_THRESHOLD_SWEEP = False
RUN_GRID_SEARCH = False

SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.0
TEST_RATIO = 0.2

PREPROCESS_CONFIG = BACKEND_COMPAT_PREPROCESS_CONFIG

LBPH_PARAMS = {
    "radius": 2,
    "neighbors": 8,
    "grid_x": 7,
    "grid_y": 7,
    "threshold": None,
}

STAGED_RESUME_MODE = "rebuild"
STAGES = [
    {"stage_name": "pretrain", "stage_train_dir": "TestData/Faces_train"},
    # {"stage_name": "main", "stage_train_dir": "datasets/main_train"},
    # {"stage_name": "finetune", "stage_manifest": "metadata/finetune_manifest.csv"},
]
THRESHOLD_SWEEP_VALUES = [40, 50, 60, 70, 80, 90, 100]
GRID_SEARCH_PARAMS = {
    "radius": [1, 2],
    "neighbors": [8],
    "grid_x": [7, 8],
    "grid_y": [7, 8],
    "threshold": [None],
}

PREVIEW_SAMPLE_COUNT = 8
ERROR_SAMPLE_COUNT = 12
SINGLE_IMAGE_PATH = None

## 3. Manifest 生成与数据概览

`RUN_SCAN_RAW=True` 时会扫描 `Faces_raw` 并写入 `metadata/manifest.csv`。否则优先读取已有 manifest。

In [ ]:
manifest_file = WORKSPACE / MANIFEST_PATH

if RUN_SCAN_RAW:
    rows = scan_faces_raw(WORKSPACE, raw_dir=RAW_DIR, out_manifest=MANIFEST_PATH)
elif manifest_file.exists():
    rows = read_manifest(manifest_file)
else:
    rows = []

manifest_df = pd.DataFrame(rows)
if manifest_df.empty:
    display(Markdown("未发现 manifest。请先放入原始数据，或启用 `RUN_SCAN_RAW=True` 后运行本单元。"))
else:
    summary = pd.DataFrame(
        {
            "metric": ["images", "identities", "splits", "quality_flags"],
            "value": [
                len(manifest_df),
                manifest_df["identity"].nunique(),
                ", ".join(sorted(set(manifest_df["split"].fillna("").astype(str)))) or "unsplit",
                ", ".join(sorted(set(manifest_df["quality_flag"].fillna("").astype(str)))) or "unknown",
            ],
        }
    )
    display(summary)
    display(manifest_df.head(10))

In [ ]:
if not manifest_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    manifest_df["identity"].value_counts().sort_values(ascending=False).head(30).plot(kind="bar", ax=axes[0], title="Top identities by samples")
    manifest_df["split"].replace("", "unsplit").value_counts().plot(kind="bar", ax=axes[1], title="Split distribution")
    manifest_df["quality_flag"].replace("", "unknown").value_counts().head(20).plot(kind="bar", ax=axes[2], title="Quality flags")
    for axis in axes:
        axis.tick_params(axis="x", labelrotation=60)
    plt.tight_layout()
else:
    display(Markdown("数据概览跳过：manifest 为空。"))

## 4. 分层切分

`RUN_SPLIT=True` 时会按身份分层切分，并写入 `Faces_train`、`Faces_val`、`Faces_test`。

In [ ]:
if RUN_SPLIT:
    split_rows = stratified_split(
        WORKSPACE,
        manifest=MANIFEST_PATH,
        train_dir=TRAIN_DIR,
        val_dir=VAL_DIR,
        test_dir=TEST_DIR,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED,
        copy_files=True,
    )
    manifest_df = pd.DataFrame(split_rows)
    display(manifest_df.groupby(["identity", "split"]).size().unstack(fill_value=0).head(20))
else:
    display(Markdown("分层切分未执行。需要切分时设置 `RUN_SPLIT=True`。"))

## 5. 预处理可视化

抽样查看原图与 LBPH 输入矩阵，重点检查灰度化、裁剪、resize、CLAHE、未检测到人脸时的 fallback 状态。

In [ ]:
def resolve_image_path(relative_or_absolute):
    path = Path(str(relative_or_absolute))
    return path if path.is_absolute() else WORKSPACE / path


if manifest_df.empty:
    display(Markdown("预处理预览跳过：manifest 为空。"))
else:
    preview_rows = manifest_df.sample(
        n=min(PREVIEW_SAMPLE_COUNT, len(manifest_df)),
        random_state=SEED,
    ).to_dict("records")
    fig, axes = plt.subplots(len(preview_rows), 2, figsize=(8, 3 * len(preview_rows)))
    if len(preview_rows) == 1:
        axes = [axes]
    for axis_row, row in zip(axes, preview_rows):
        image_path = resolve_image_path(row["relative_path"])
        result = preprocess_image(image_path, PREPROCESS_CONFIG)
        axis_row[0].imshow(Image.open(image_path).convert("RGB"))
        axis_row[0].set_title(f"raw: {row['identity']}")
        axis_row[0].axis("off")
        axis_row[1].imshow(result.face, cmap="gray")
        axis_row[1].set_title(f"processed: {result.status}")
        axis_row[1].axis("off")
    plt.tight_layout()

## 6. Multi-stage training

`RUN_STAGED_TRAINING=True` runs the configured `STAGES` in order. Use `STAGED_RESUME_MODE="rebuild"` for reproducible benchmark models, or `"update"` for fast incremental LBPH continuation from the latest checkpoint.

In [ ]:
if RUN_STAGED_TRAINING:
    staged_results = []
    for stage in STAGES:
        result = run_training_stage(
            workspace=WORKSPACE,
            stage_name=stage["stage_name"],
            stage_manifest=stage.get("stage_manifest"),
            stage_train_dir=stage.get("stage_train_dir"),
            algorithm_dir=ALGORITHM_DIR,
            resume_mode=stage.get("resume_mode", STAGED_RESUME_MODE),
            run_id=stage.get("run_id"),
            preprocess_config=PREPROCESS_CONFIG,
            evaluate_after_stage=stage.get("evaluate_after_stage", False),
            test_dir=stage.get("test_dir", TEST_DIR),
            reports_dir=REPORTS_DIR,
            **LBPH_PARAMS,
        )
        staged_results.append(result)
    display(pd.DataFrame(staged_results))
else:
    training_state_path = WORKSPACE / ALGORITHM_DIR / "training_state.json"
    if training_state_path.exists():
        state = load_training_state(training_state_path)
        display(pd.DataFrame(state.get("stages", [])))
    else:
        display(Markdown("Multi-stage training skipped. Set `RUN_STAGED_TRAINING=True` to run stages."))

## 6. LBPH 训练

`RUN_TRAIN=True` 时训练 OpenCV LBPH，并保存 `Algorithm/face_recognizer_model.xml`、`label_mapping.json`、`preprocess_config.json`、`training_report.json`。

In [ ]:
training_report_path = WORKSPACE / ALGORITHM_DIR / "training_report.json"

if RUN_TRAIN:
    training_report = train_lbph(
        workspace=WORKSPACE,
        manifest=MANIFEST_PATH if manifest_file.exists() else None,
        train_dir=TRAIN_DIR,
        algorithm_dir=ALGORITHM_DIR,
        preprocess_config=PREPROCESS_CONFIG,
        **LBPH_PARAMS,
    )
elif training_report_path.exists():
    training_report = json.loads(training_report_path.read_text(encoding="utf-8"))
else:
    training_report = None

if training_report:
    display(pd.json_normalize(training_report).T.rename(columns={0: "value"}))
else:
    display(Markdown("训练未执行且没有发现已有训练报告。需要训练时设置 `RUN_TRAIN=True`。"))

## 7. 测试集评估

`RUN_EVALUATE=True` 时读取 `Faces_test` 并写出 `reports/metrics.json`、混淆矩阵、每人准确率和错误样本。

In [ ]:
metrics_path = WORKSPACE / REPORTS_DIR / "metrics.json"

if RUN_EVALUATE:
    metrics = evaluate_directory(
        test_dir=WORKSPACE / TEST_DIR,
        algorithm_dir=WORKSPACE / ALGORITHM_DIR,
        reports_dir=WORKSPACE / REPORTS_DIR,
        threshold=LBPH_PARAMS["threshold"],
    )
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
else:
    metrics = None

if metrics:
    display(pd.DataFrame([{
        "overall_accuracy": metrics.get("overall_accuracy"),
        "macro_precision": metrics.get("macro_precision"),
        "macro_recall": metrics.get("macro_recall"),
        "macro_f1": metrics.get("macro_f1"),
        "num_test_images": metrics.get("num_test_images"),
        "num_non_ok_predictions": metrics.get("num_non_ok_predictions"),
    }]))
else:
    display(Markdown("评估未执行且没有发现已有评估报告。需要评估时设置 `RUN_EVALUATE=True`。"))

## Training adjustment helpers

Optional threshold sweep and lightweight LBPH grid search write reports under `reports/`. Keep these switches off for benchmark packaging runs.

In [ ]:
if RUN_THRESHOLD_SWEEP:
    threshold_sweep = run_threshold_sweep(
        test_dir=WORKSPACE / TEST_DIR,
        algorithm_dir=WORKSPACE / ALGORITHM_DIR,
        reports_dir=WORKSPACE / REPORTS_DIR / "threshold_sweep",
        thresholds=THRESHOLD_SWEEP_VALUES,
    )
    display(pd.DataFrame(threshold_sweep["results"]))
    display(threshold_sweep["best"])
else:
    display(Markdown("Threshold sweep skipped. Set `RUN_THRESHOLD_SWEEP=True` to run it."))

if RUN_GRID_SEARCH:
    grid_search = grid_search_lbph(
        workspace=WORKSPACE,
        manifest=MANIFEST_PATH,
        test_dir=TEST_DIR,
        output_dir=WORKSPACE / REPORTS_DIR / "grid_search",
        param_grid=GRID_SEARCH_PARAMS,
        preprocess_config=PREPROCESS_CONFIG,
    )
    display(pd.DataFrame(grid_search["results"]))
    display(grid_search["best"])
else:
    display(Markdown("Grid search skipped. Set `RUN_GRID_SEARCH=True` to run it."))

In [ ]:
if metrics and metrics.get("confusion_matrix"):
    confusion_df = pd.DataFrame(metrics["confusion_matrix"]).fillna(0).astype(int).T
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(confusion_df.columns)), max(6, 0.35 * len(confusion_df))))
    image = ax.imshow(confusion_df.values, cmap="Blues")
    ax.set_xticks(range(len(confusion_df.columns)))
    ax.set_yticks(range(len(confusion_df.index)))
    ax.set_xticklabels(confusion_df.columns, rotation=90)
    ax.set_yticklabels(confusion_df.index)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
else:
    display(Markdown("混淆矩阵跳过：没有可用评估结果。"))

In [ ]:
if metrics:
    per_identity_df = pd.DataFrame(
        sorted(metrics.get("per_identity_accuracy", {}).items()),
        columns=["identity", "accuracy"],
    )
    error_df = pd.DataFrame(metrics.get("error_cases", []))
    display(per_identity_df.sort_values("accuracy").head(30))
    display(error_df.head(50))
else:
    display(Markdown("每人准确率和错误样本跳过：没有可用评估结果。"))

## 8. 错误样本回看

从 `error_cases` 中抽样展示原图和预处理结果，用于定位光照、虚焦、侧脸、标签映射或阈值问题。

In [ ]:
if metrics and metrics.get("error_cases"):
    error_rows = metrics["error_cases"][:ERROR_SAMPLE_COUNT]
    fig, axes = plt.subplots(len(error_rows), 2, figsize=(8, 3 * len(error_rows)))
    if len(error_rows) == 1:
        axes = [axes]
    for axis_row, row in zip(axes, error_rows):
        image_path = Path(row["image_path"])
        result = preprocess_image(image_path, PREPROCESS_CONFIG)
        axis_row[0].imshow(Image.open(image_path).convert("RGB"))
        axis_row[0].set_title(f"true={row['true_label']} pred={row['predicted_label']}")
        axis_row[0].axis("off")
        axis_row[1].imshow(result.face, cmap="gray")
        axis_row[1].set_title(f"status={row['status']} confidence={row['confidence']}")
        axis_row[1].axis("off")
    plt.tight_layout()
else:
    display(Markdown("没有错误样本可回看。"))

## 9. 单图预测

把 `SINGLE_IMAGE_PATH` 设置为图片路径后运行本单元，可快速检查单张图片的预测标签、confidence 和状态。

In [ ]:
if SINGLE_IMAGE_PATH:
    predictor = LBPHPredictor(
        algorithm_dir=WORKSPACE / ALGORITHM_DIR,
        threshold=LBPH_PARAMS["threshold"],
    )
    single_prediction = predictor.predict_image(resolve_image_path(SINGLE_IMAGE_PATH))
    display(single_prediction)
else:
    display(Markdown("单图预测未执行。需要时设置 `SINGLE_IMAGE_PATH`。"))

## 10. 后端 artifact 与 benchmark 打包

这一步只导出离线产物，不修改后端数据库，也不把 `image_generate` 写成运行时依赖。

In [ ]:
export_results = {}

if RUN_BACKEND_EXPORT:
    export_results["backend_artifacts"] = convert_algorithm_to_backend_artifacts(
        WORKSPACE / ALGORITHM_DIR,
        WORKSPACE / BACKEND_ARTIFACT_DIR,
    )

if RUN_BENCHMARK_PACKAGE:
    export_results["benchmark_package"] = str(
        create_benchmark_package(
            workspace=WORKSPACE,
            output=BENCHMARK_ZIP,
            minimal=True,
        )
    )

if export_results:
    display(export_results)
else:
    display(Markdown("导出未执行。需要时设置 `RUN_BACKEND_EXPORT` 或 `RUN_BENCHMARK_PACKAGE`。"))